In [3]:
!pip install transformers datasets torch -q

print("done!")



done!


In [4]:
import torch
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
from torch.utils.data import DataLoader, Dataset
from datasets import load_dataset
import numpy as np
from torch.optim import AdamW

# Ελέγχουμε αν έχουμε GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [5]:
# Dataset
dataset = load_dataset("stanfordnlp/imdb")

# Tokenizer
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

print("Ready!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

...

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Ready!


In [6]:
class IMDBDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=256):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        text = self.data[idx]['text']
        label = self.data[idx]['label']

        encoding = self.tokenizer(
            text,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'label': torch.tensor(label, dtype=torch.long)
        }

print("Dataset class ready!")

Dataset class ready!


In [7]:
# Χρησιμοποιούμε subset για γρηγορότερο training
train_dataset = IMDBDataset(dataset['train'], tokenizer)
test_dataset = IMDBDataset(dataset['test'], tokenizer)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"Train batches: {len(train_loader)}")
print(f"Test batches: {len(test_loader)}")


Train batches: 782
Test batches: 782


In [8]:
model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=2
)
model = model.to(device)

print("Model ready!")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model ready!


In [9]:
optimizer = AdamW(model.parameters(), lr=2e-5)

def train_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0
    correct = 0

    for batch in loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = outputs.logits.argmax(dim=1)
        correct += (preds == labels).sum().item()

    return total_loss / len(loader), correct / len(loader.dataset)

print("Training loop ready!")

Training loop ready!


In [10]:
def evaluate(model, loader):
    model.eval()
    correct = 0

    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = outputs.logits.argmax(dim=1)
            correct += (preds == labels).sum().item()

    return correct / len(loader.dataset)

print("Evaluation loop ready!")


Evaluation loop ready!


In [11]:
EPOCHS = 2

for epoch in range(EPOCHS):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer)
    test_acc = evaluate(model, test_loader)

    print(f"Epoch {epoch+1}/{EPOCHS}")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Test Acc: {test_acc:.4f}")
    print("-"*40)

Epoch 1/2
Train Loss: 0.2867 | Train Acc: 0.8812
Test Acc: 0.9085
----------------------------------------
Epoch 2/2
Train Loss: 0.1730 | Train Acc: 0.9342
Test Acc: 0.9085
----------------------------------------


In [12]:
model.save_pretrained('./sentiment_model')
tokenizer.save_pretrained('./sentiment_model')

print("Model saved!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved!


In [13]:
def predict_bert(text):
    inputs = tokenizer(
        text,
        max_length=256,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    ).to(device)

    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=1)
        pred = probs.argmax().item()
        confidence = probs.max().item()

    label = "POSITIVE 😊" if pred == 1 else "NEGATIVE 😞"
    print(f"Text: {text}")
    print(f"Sentiment: {label}")
    print(f"Confidence: {confidence:.2%}\n")

predict_bert("This movie was absolutely amazing, I loved every minute of it!")
predict_bert("Terrible film, complete waste of time. Very boring and predictable.")
predict_bert("It was okay, nothing special but not bad either.")

Text: This movie was absolutely amazing, I loved every minute of it!
Sentiment: POSITIVE 😊
Confidence: 99.53%

Text: Terrible film, complete waste of time. Very boring and predictable.
Sentiment: NEGATIVE 😞
Confidence: 99.49%

Text: It was okay, nothing special but not bad either.
Sentiment: NEGATIVE 😞
Confidence: 64.86%



In [14]:
from google.colab import files
import shutil

shutil.make_archive('sentiment_model', 'zip', '.', 'sentiment_model')
files.download('sentiment_model.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>